# Adaption — Sensitivität gegenüber dem Suchraum

Schreibt die Makros für die Anhangstabelle zur Sensitivitätsprüfung nach `results/adaptation_sensitivity.tex`. Es wird nichts gerechnet: gelesen werden die gesicherten Caches des abgebrochenen Laufs `grenzen-in-chunks` unter `results/sensitivity/` und zum Vergleich das Screening des Endstands aus `results/<fall>_adaptation.json`.

- **Randsonde:** Konfiguration des Endstands, ein Parameter über die alte Grenze hinaus variiert, voller Strom, fünf Seeds.
- **Screening:** Detektoren im geweiteten Suchraum neu abgestimmt, voller Strom, fünf Seeds.

## Konstanten

`CASES` nennt je Fall das Makro-Präfix, die Cache-Namen, die Ergebnisdatei des Endstands und die Nachkommastellen des RMSE. `TIE_ORDER` ist die Reihenfolge der Gleichstandsregel aus den Adaptions-Notebooks. Makronamen dürfen keine Ziffern enthalten, die Stufen einer Sonde heißen deshalb `A`, `B`, …; `A` ist der Wert des Endstands.

In [1]:
CASES = {
    "syn": {"prefix": "sensSyn", "probe": "syn_adapt_probe_1393f52a",
            "screening": "syn_adapt_screening_99281e8b", "result": "syn_adaptation.json", "nd": 3},
    "tep": {"prefix": "sensTep", "probe": "tep_adapt_probe_e5f2ad89",
            "screening": "tep_adapt_screening_84777c4a", "result": "tep_adaptation.json", "nd": 4},
}
TIE_ORDER = ("RMSE", "DDM", "EDDM", "ADWIN", "KSWIN")
OUT_NAME = "adaptation_sensitivity.tex"

STRATEGY_NAMES = {"blind": "Blind", "informed": "Informed", "combined": "Combined",
                  "blind_rec": "BlindRec", "informed_rec": "InformedRec", "combined_rec": "CombinedRec"}
PARAM_NAMES = {"blind_window": "Window", "window_prev": "WindowPrev", "window_post": "WindowPost",
               "rec_max": "RecMax", "err_tol": "ErrTol", "det_min_num_instances": "MinInst"}
WINDOW_PARAMS = ("blind_window", "window_prev", "window_post")
DETECTOR_NAMES = {"KSWIN": "Kswin", "ADWIN": "Adwin", "DDM": "Ddm", "EDDM": "Eddm", "RMSE": "Threshold"}
DETECTOR_LABELS = {"RMSE": "Schwellwert-Detektor", "DDM": r"\ac{DDM}", "EDDM": r"\ac{EDDM}",
                   "ADWIN": r"\ac{ADWIN}", "KSWIN": r"\ac{KSWIN}"}
LEVELS = "ABCDEFGH"

## Pfade

Das Notebook liegt unter `_claude_output/werkzeuge/`, die Repo-Wurzel ist zwei Ebenen darüber (oder `BASE_DIR`).

In [ ]:
import json
import os
import pickle
import sys
from pathlib import Path

import pandas as pd
from scipy import stats

base_dir = Path(os.environ.get("BASE_DIR", Path.cwd().parents[0])).resolve()
sys.path.insert(0, str(base_dir))
from src.utils import latex_export as lx

results_dir = base_dir / "results"
sens_dir = results_dir / "sensitivity"

## Laden

Jeder Cache besteht aus der Nutzlast (`.pkl`) und der Begleitdatei (`.json`) mit Konfiguration und Code-Stand. Alle Caches müssen vom selben Code-Stand stammen.

In [3]:
def load_cache(stem):
    with open(sens_dir / f"{stem}.pkl", "rb") as f:
        payload = pickle.load(f)
    doc = json.loads((sens_dir / f"{stem}.json").read_text(encoding="utf-8"))
    return payload, doc


loaded = {}
for case, spec in CASES.items():
    probes, probe_doc = load_cache(spec["probe"])
    screening, screening_doc = load_cache(spec["screening"])
    final = json.loads((results_dir / spec["result"]).read_text(encoding="utf-8"))
    loaded[case] = {"probes": probes, "probe_cfg": probe_doc["config"]["adapt_probe"],
                    "screening": screening, "final": final,
                    "code": {probe_doc["code_version"], screening_doc["code_version"]}}

code_versions = set().union(*(entry["code"] for entry in loaded.values()))
assert len(code_versions) == 1, code_versions
CODE_VERSION = code_versions.pop()
print(f"Code-Stand der Caches: {CODE_VERSION}")

Code-Stand der Caches: ae0cd92


## Randsonde

Je Sonde eine Zeile pro Stufe mit Mittelwert und Streuung über die Seeds. `p_value` ist der gepaarte t-Test gegen den Wert des Endstands (Stufe `A`). Fensterlängen werden zusätzlich in Tagen angegeben: Wert auf dem Tuningraster mal `rescale`, geteilt durch die Chunk-Länge des Zielrasters.

In [4]:
def probe_rows(case):
    cfg = loaded[case]["probe_cfg"]
    rows = []
    for probe in loaded[case]["probes"]:
        keys = list(probe["table"])
        base_seeds = probe["table"][keys[0]]["rmse_seeds"]
        for level, value, key in zip(LEVELS, probe["values"], keys):
            entry = probe["table"][key]
            is_window = probe["param"] in WINDOW_PARAMS
            rows.append({
                "case": case, "strategy": probe["strategy"], "param": probe["param"], "level": level,
                "value": value,
                "days": value * cfg["rescale"] / cfg["chunk"] if is_window else None,
                "rmse": entry["rmse"], "rmse_std": entry["rmse_std"],
                "p_value": None if level == "A" else float(stats.ttest_rel(entry["rmse_seeds"], base_seeds).pvalue),
                "binding": probe["binding"],
            })
    return rows


probe_table = pd.DataFrame([row for case in CASES for row in probe_rows(case)])
probe_table

,case,strategy,param,level,value,days,rmse,rmse_std,p_value,binding
0,syn,blind,blind_window,A,377.000000,15.708333,0.642285,0.080022,NaN,True
1,syn,blind,blind_window,B,566.000000,23.583333,0.683335,0.054021,0.458447,True
2,syn,blind,blind_window,C,754.000000,31.416667,0.750987,0.137862,0.128012,True
3,syn,blind,blind_window,D,1508.000000,62.833333,0.554173,0.040935,0.041220,True
4,syn,blind_rec,blind_window,A,377.000000,15.708333,0.576595,0.031671,NaN,False
5,syn,blind_rec,blind_window,B,566.000000,23.583333,0.584058,0.061927,0.779801,False
6,syn,blind_rec,blind_window,C,754.000000,31.416667,0.595818,0.037400,0.262848,False
7,syn,blind_rec,blind_window,D,1508.000000,62.833333,0.573740,0.010321,0.869324,False
8,syn,informed_rec,rec_max,A,21.000000,NaN,0.540980,0.022275,NaN,False
9,syn,informed_rec,rec_max,B,14.000000,NaN,0.721875,0.111444,0.027028,False


## Screening

Stellt je Detektor den RMSE des Endstands dem im geweiteten Suchraum gegenüber und wendet die Gleichstandsregel mit den Schranken aus der Sonden-Konfiguration erneut an.

In [5]:
def screening_summary(case):
    cfg = loaded[case]["probe_cfg"]
    screening = loaded[case]["screening"]
    lead = min(screening, key=lambda det: screening[det]["rmse"])
    tied = [lead]
    for det in screening:
        if det == lead:
            continue
        p = stats.ttest_rel(screening[det]["rmse_seeds"], screening[lead]["rmse_seeds"]).pvalue
        rel = screening[det]["rmse"] / screening[lead]["rmse"] - 1.0
        if p > cfg["tie_alpha"] and rel <= cfg["tie_max_rel"]:
            tied.append(det)
    return {"lead": lead, "tied": tied, "chosen": min(tied, key=TIE_ORDER.index)}


screening_table = pd.DataFrame([
    {"case": case, "detector": det,
     "rmse_final": loaded[case]["final"]["screening"][det]["rmse"],
     "rmse": entry["rmse"], "rmse_std": entry["rmse_std"], "n_train": entry["n_train"]}
    for case in CASES for det, entry in loaded[case]["screening"].items()])
screening_choice = {case: screening_summary(case) for case in CASES}
print({case: (loaded[case]["final"]["informed_detector"], "->", choice["chosen"])
       for case, choice in screening_choice.items()})
screening_table

{'syn': ('KSWIN', '->', 'RMSE'), 'tep': ('KSWIN', '->', 'RMSE')}


,case,detector,rmse_final,rmse,rmse_std,n_train
0,syn,KSWIN,0.784815,0.608984,0.001362,79.0
1,syn,ADWIN,0.845452,0.919912,0.020391,35.8
2,syn,DDM,1.669732,0.680607,0.009014,265.2
3,syn,EDDM,1.500277,0.857499,0.012349,284.0
4,syn,RMSE,0.890264,0.609415,0.004283,25.8
5,tep,KSWIN,0.021511,0.038031,0.004096,498.4
6,tep,ADWIN,0.023988,0.020619,0.000526,691.2
7,tep,DDM,0.025191,0.024034,0.000464,631.0
8,tep,EDDM,0.064251,0.036567,0.004216,405.4
9,tep,RMSE,0.057039,0.020989,0.000984,622.4


## Makros schreiben

Makronamen: `<Präfix><Strategie><Parameter><Stufe><Größe>` für die Sonde, `<Präfix>Screen<Detektor><Größe>` für das Screening.

In [6]:
mx = lx.MacroExport("automatisch erzeugt aus adaptation_sensitivity.ipynb - nicht manuell editieren")
mx.comment("Herkunft")
mx.raw("sensCodeVersion", CODE_VERSION)

for case, spec in CASES.items():
    prefix, nd = spec["prefix"], spec["nd"]
    cfg = loaded[case]["probe_cfg"]

    mx.comment(f"{case}: Randsonde (Endstand-Konfiguration, ein Parameter variiert)")
    mx.param(f"{prefix}NSeeds", len(cfg["seeds"]), spell_out=True)
    for (strategy, param), group in probe_table[probe_table["case"] == case].groupby(["strategy", "param"], sort=False):
        stem = f"{prefix}{STRATEGY_NAMES[strategy]}{PARAM_NAMES[param]}"
        mx.raw(f"{stem}Binding", "bindet" if group["binding"].iloc[0] else "bindet nicht")
        for row in group.itertuples():
            if float(row.value).is_integer():
                mx.integer(f"{stem}{row.level}Value", row.value)
            else:
                mx.num(f"{stem}{row.level}Value", row.value, 2)
            if row.param in WINDOW_PARAMS:
                mx.num(f"{stem}{row.level}Days", row.days, 0)
            mx.num(f"{stem}{row.level}Rmse", row.rmse, nd)
            mx.num(f"{stem}{row.level}Std", row.rmse_std, nd)
            if row.level != "A":
                mx.num(f"{stem}{row.level}PValue", row.p_value, 3)

    mx.comment(f"{case}: Screening im geweiteten Suchraum gegen Endstand")
    choice = screening_choice[case]
    mx.raw(f"{prefix}ScreenLead", DETECTOR_LABELS[choice["lead"]])
    mx.raw(f"{prefix}ScreenTied", ", ".join(DETECTOR_LABELS[det] for det in choice["tied"]))
    mx.raw(f"{prefix}ScreenDetector", DETECTOR_LABELS[choice["chosen"]])
    for row in screening_table[screening_table["case"] == case].itertuples():
        stem = f"{prefix}Screen{DETECTOR_NAMES[row.detector]}"
        mx.num(f"{stem}Final", row.rmse_final, nd)
        mx.num(f"{stem}Rmse", row.rmse, nd)
        mx.num(f"{stem}Std", row.rmse_std, nd)
        mx.num(f"{stem}NTrain", row.n_train, 0)

out_path = results_dir / OUT_NAME
out_path.write_text(mx.render(), encoding="utf-8")
print(f"LaTeX-Makros -> {out_path}\n\n{mx.render()}")

LaTeX-Makros -> C:\git\process_optimization_concept_drift\results\adaptation_sensitivity.tex

% automatisch erzeugt aus adaptation_sensitivity.ipynb - nicht manuell editieren

% Herkunft
\newcommand{\sensCodeVersion}{ae0cd92}

% syn: Randsonde (Endstand-Konfiguration, ein Parameter variiert)
\newcommand{\sensSynNSeeds}{fünf}
\newcommand{\sensSynBlindWindowBinding}{bindet}
\newcommand{\sensSynBlindWindowAValue}{377}
\newcommand{\sensSynBlindWindowADays}{16}
\newcommand{\sensSynBlindWindowARmse}{0{,}642}
\newcommand{\sensSynBlindWindowAStd}{0{,}080}
\newcommand{\sensSynBlindWindowBValue}{566}
\newcommand{\sensSynBlindWindowBDays}{24}
\newcommand{\sensSynBlindWindowBRmse}{0{,}683}
\newcommand{\sensSynBlindWindowBStd}{0{,}054}
\newcommand{\sensSynBlindWindowBPValue}{0{,}458}
\newcommand{\sensSynBlindWindowCValue}{754}
\newcommand{\sensSynBlindWindowCDays}{31}
\newcommand{\sensSynBlindWindowCRmse}{0{,}751}
\newcommand{\sensSynBlindWindowCStd}{0{,}138}
\newcommand{\sensSynBlindWindowCPValue}